In [1]:
import pandas as pd #tabular data, data frame
import numpy as np
import joblib
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import ConvergenceWarning
import time 

Xtrain = pd.read_csv('x_train.csv')
Xtest = pd.read_csv('x_test.csv')
ytrain = pd.read_csv('y_train.csv')
ytest = pd.read_csv('y_test.csv')

In [2]:
# results is okay but not the best 
from sklearn.metrics import mean_squared_error, r2_score
import sklearn
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from IPython.display import display, HTML
sklearn.set_config(display='diagram')

In [3]:
# Record starting time
start = time.perf_counter()

# 1. Initialize the XGBoost Regressor
# No feature scaling required for tree-based models like XGBoost
xgb_regressor = xgb.XGBRegressor(
    n_estimators=100,     # Number of sequential trees to build
    learning_rate=0.1,    # Step size shrinkage used to prevent overfitting
    max_depth=5,          # Maximum depth of each individual tree
    random_state=42,      # Ensures reproducible results
    n_jobs=-1             # Uses all available CPU cores for faster execution
)

# 2. Train (fit) the model on training data
xgb_regressor.fit(Xtrain, ytrain)

# 3. Make predictions on the test set
y_pred = xgb_regressor.predict(Xtest)

# 4. Evaluate model performance using R-squared and Mean Squared Error
train_r2 = xgb_regressor.score(Xtrain, ytrain)
test_r2 = xgb_regressor.score(Xtest, ytest)

# Record ending time
end = time.perf_counter()

print(f"XGBoost Train R2 Score: {train_r2:.4f}")
print(f"XGBoost Test R2 Score:  {test_r2:.4f}")
print(f"Execution time: {end - start:.4f} seconds")


XGBoost Train R2 Score: 1.0000
XGBoost Test R2 Score:  0.9996
Execution time: 0.3166 seconds


In [5]:
# ==========================================
# 2. Define Time-Series Cross-Validation
# ==========================================
# TimeSeriesSplit ensures walk-forward validation.
tscv = TimeSeriesSplit(n_splits=5)

# ==========================================
# 3. Define Hyperparameter Grid & Base Model
# ==========================================
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],          # Number of boosting trees
    'max_depth': [3, 5, 7],                  # Tree depth (keep low for noisy data)
    'learning_rate': [0.01, 0.05, 0.1],      # Step size shrinkage
    'subsample': [0.7, 0.8, 1.0],            # Fraction of sample rows per tree
    'colsample_bytree': [0.7, 0.8, 1.0],     # Fraction of features per tree
    'min_child_weight': [1, 5],
    'reg_alpha': [0, 0.1, 1.0],              # L1 regularization (Lasso penalty)
    'reg_lambda': [1.0, 5.0, 10.0]           # L2 regularization (Ridge penalty)
}

# ==========================================
# 4. Set Up and Run GridSearchCV
# ==========================================
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=TimeSeriesSplit(n_splits=5),          # Use TimeSeriesSplit!
    scoring='neg_mean_squared_error',        # Optimize for lower MSE
    n_jobs=-1,                               # Parallel execution across all cores
    verbose=1
)

print("Starting Grid Search...")
grid_search.fit(Xtrain, ytrain)

# ==========================================
# 5. Review Best Parameters & Performance
# ==========================================
print("\n" + "="*40)
print("BEST HYPERPARAMETERS FOUND:")
print("="*40)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")
    
# 1. Define the exact parameters you want to see
selected_params = [
    'n_estimators', 
    'max_depth', 
    'learning_rate', 
    'subsample', 
    'colsample_bytree', 
    'min_child_weight',
    'reg_alpha',
    'reg_lambda',
]

# 2. Extract values from best_estimator_
all_params = grid_search.best_estimator_.get_params()
filtered_params = {param: all_params[param] for param in selected_params if param in all_params}

# 3. Format as a clean, styled HTML box similar to the diagram
df = pd.DataFrame(list(filtered_params.items()), columns=['Parameter', 'Value'])

# Save best model
best_xgb_model = grid_search.best_estimator_
joblib.dump(best_xgb_model, 'xgboost.pkl')

print("Best model successfully saved to xgboost.pkl!")

# Render in Jupyter notebook
display(HTML(df.to_html(index=False)))

Starting Grid Search...
Fitting 5 folds for each of 4374 candidates, totalling 21870 fits

BEST HYPERPARAMETERS FOUND:
colsample_bytree: 0.7
learning_rate: 0.05
max_depth: 5
min_child_weight: 5
n_estimators: 200
reg_alpha: 0.1
reg_lambda: 1.0
subsample: 0.8
Best model successfully saved to xgboost.pkl!


Parameter,Value
n_estimators,200.00
max_depth,5.00
learning_rate,0.05
subsample,0.80
colsample_bytree,0.70
min_child_weight,5.00
reg_alpha,0.10
reg_lambda,1.00


In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Best model from GridSearchCV
best_model = grid_search.best_estimator_

# Predictions
ytrain_pred = best_model.predict(Xtrain)
ytest_pred = best_model.predict(Xtest)

# R²
train_r2 = best_model.score(Xtrain, ytrain)
test_r2 = best_model.score(Xtest, ytest)

# RMSE
train_rmse = np.sqrt(mean_squared_error(ytrain, ytrain_pred))
test_rmse = np.sqrt(mean_squared_error(ytest, ytest_pred))

# MAE
train_mae = mean_absolute_error(ytrain, ytrain_pred)
test_mae = mean_absolute_error(ytest, ytest_pred)

# Display results
print(f"Training R²: {train_r2:.4f}")
print(f"Testing R² : {test_r2:.4f}")

print(f"Training RMSE: {train_rmse:.4f}")
print(f"Testing RMSE : {test_rmse:.4f}")

print(f"Training MAE: {train_mae:.4f}")
print(f"Testing MAE : {test_mae:.4f}")

Training R²: 0.9999
Testing R² : 0.9997
Training RMSE: 192.2282
Testing RMSE : 371.0959
Training MAE: 102.9766
Testing MAE : 174.7812
